# CNNs with Keras on landscape data

## Checking GPU Usage

Go to the `Runtime > Change runtime type` menu and check that we're on Python 3 and that the hardware accelerator is set to "GPU".

In [ ]:
!nvidia-smi

## Get data from a git repo

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s landscape
!ls landscape
print("***")
!ls -l landscape/seg_train
print("***")
!ls -l landscape/seg_pred

## Imports

In [ ]:
import itertools
import os
import pathlib
import random
import typing

import cv2
import matplotlib
import matplotlib.pyplot as plt
import numpy
import pandas
import PIL
import seaborn
import skimage.transform
import sklearn.utils
import sklearn.metrics
import tensorflow as tf
import keras as keras
import tqdm.notebook

## Preparing the Data

To load our data, we'll combine several libraries: Pillow, NumPy & TensorFlow.

In [ ]:
INPUT_SHAPE = (150, 150, 3)


label_names = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
label_to_index = {l: i for i, l in enumerate(label_names)}


def get_images(dir_path: pathlib.Path,
               size: int = INPUT_SHAPE[0],
               channels_first: bool = False,
               shuffle: bool = True,
               create_labels: bool = True,
               ) -> typing.Tuple[tf.Tensor, tf.Tensor]:
  images = []
  if create_labels:
    labels = []

  # Iterate over the subfolders of the root: each one matches a
  # class
  for subdir_path in tqdm.notebook.tqdm(
      list(dir_path.iterdir()), desc="Processing folders"):

    dir_name = subdir_path.name

    if create_labels:
      # Assign the right label based on the folder name "labels"
      label = label_to_index.get(dir_name)

    # Add each image of the current label (folder) to our dataset
    for image_path in tqdm.notebook.tqdm(
        list(subdir_path.iterdir()), desc=f"Folder {dir_name}", leave=False):
      # Use PIL to load the image
      images.append(
          numpy.array(PIL.Image.open(image_path).resize((size, size))))
      if create_labels:
        labels.append(label)

  images = tf.constant(numpy.array(images))
  if create_labels:
    labels = tf.constant(numpy.array(labels))

  if shuffle:
    perm = tf.random.shuffle(tf.range(images.shape[0]))
    images = tf.gather(images, perm)
    if create_labels:
      labels = tf.gather(labels, perm)

  if channels_first:
    images = tf.transpose(images, perm=(0, 3, 1, 2))

  if create_labels:
    return images, labels
  else:
    return images

## `get_images` call

In [ ]:
images, labels = get_images(
    pathlib.Path("landscape") / "seg_train")

In [ ]:
print(f"Images shape:  {images.shape}")
print(f"Labels shape:  {labels.shape}")

seaborn.countplot(x=labels)
plt.title("Counts of the different labels")
plt.ylabel("Count")
plt.xlabel("Label")
plt.show()

In [ ]:
# Creation of the subplot grid. The argument figsize is given to enlarge
# the size of the figure which is small by default
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# We choose 25 indices at random, without replacement (we don't want to 
# display the same image twice)
random_indexes = numpy.random.choice(images.shape[0],
                                     size=(5, 5),
                                     replace=False)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    image = images[img_index]
    label = label_names[labels[img_index]]

    # matplotlib's imshow is usefull in computer vision works.
    # vision
    ax[i, j].imshow(image)
    ax[i, j].set_title(f"Example {img_index} ({label})")
    ax[i, j].axis('off')

## Model creation

Here is an example of a "minimalist" CNN

In [ ]:
# Model initialization and definition

# The model is a stack of layers where the data flows sequentially
model = keras.models.Sequential()
# The first layer specifies the input size
model.add(keras.layers.Input(shape=(150, 150, 3)))
# A first layer of neurons with 1 convolution of 3x3 pixels
model.add(keras.layers.Conv2D(1,
                              kernel_size=(3, 3),
                              activation="relu"))
# A max pooling layer
model.add(keras.layers.MaxPool2D(3,3))

# A tensor manipulation layer: removes all dimensions
# except the batch one and another one holding all the values
model.add(keras.layers.Flatten())

# A dense output layer with 6 neurons and a softmax activation
model.add(keras.layers.Dense(6, activation="softmax"))

# Compile the model, defining the loss function
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# Display a summary of the model
model.summary()

## Can you explain the different numbers of parameters?

### Solution


First layer of 1 convolutions: (kernel size) * (nb kernel) * (nb input channels) + (nb bias (= nb kernel)) = (3 * 3 ) * 1 * 3 + 1

Last dense layer: (input dim) * (output dim) + (nb bias) = 2401 * 6 + 6



## Training

Let's learn this model on our data! First, we train on a single epoch to simply check that our model is operational.

In [ ]:
# Train the model
training_history = model.fit(images, labels, epochs=1, validation_split=0.30)

## Improve this performance

Build on the previous model by adding layers, making the layers smaller or bigger.

Aim for 10-20 iterations and less than 1 minute per iteration (for obvious reasons).

Consider using a dropout layer just before the last dense layer to improve smoothness.

Accuracy above 70% on the validation basis can be achieved in a reasonable time.

The proposed solution takes $\approx$45 seconds per iteration for 15 iterations and achieves around 85% accuracy on the validation basis.

In [ ]:
# Your improvements here
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=(150, 150, 3)))
model.add(keras.layers.Conv2D(10,
                              kernel_size=(3, 3),
                              activation="relu"))
model.add(keras.layers.MaxPool2D(3,3))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(6, activation="softmax"))
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# Display a summary of the model
model.summary()

# Train the model
training = model.fit(images, labels, epochs=10, validation_split=0.30)


# Plot the training metrics
def plot_metrics(history) -> None:
  plt.plot(training.history["accuracy"])
  plt.plot(training.history["val_accuracy"])
  plt.title("Model accuracy")
  plt.ylabel("Accuracy")
  plt.xlabel("Epoch")
  plt.legend(["Training", "Validation"], loc="upper left")
  plt.show()

  plt.plot(training.history["loss"])
  plt.plot(training.history["val_loss"])
  plt.title("Model loss")
  plt.ylabel("Loss")
  plt.xlabel("Epoch")
  plt.legend(["Training", "Validation"], loc="upper right")
  plt.show()


plot_metrics(training.history)

### Solution

In [ ]:
def conv() -> keras.layers.Conv2D:
  return keras.layers.Conv2D(filters=200,
                             kernel_size=3,
                             activation="relu",
                             kernel_initializer="orthogonal",
                             padding="same")


def pooling() -> keras.layers.MaxPooling2D:
  return keras.layers.MaxPooling2D(2, 2, padding="same")


def dropout(rate: float = 0.2) -> keras.layers.Dropout:
  return keras.layers.Dropout(rate)

def dense(units: int, activation: str = "relu") -> keras.layers.Dense:
  return keras.layers.Dense(units, activation=activation)


model = keras.models.Sequential([
    keras.layers.Input(shape=(150, 150, 3)),
    conv(),
    pooling(),
    conv(),
    pooling(),
    conv(),
    pooling(),
    conv(),
    pooling(),
    conv(),
    pooling(),
    conv(),
    pooling(),
    conv(),
    keras.layers.Flatten(),
    dropout(),
    dense(200),
    dropout(),
    dense(100),
    dropout(),
    dense(50),
    dropout(),
    dense(6, activation="softmax"),
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# Display a summary of the model
model.summary()

In [ ]:
# Training
training = model.fit(images,
                     labels,
                     epochs=15,
                     validation_split=0.30,
                     batch_size=128)

# Display training metrics over time
plot_metrics(training.history)

An implementation of [LeNet](https://en.wikipedia.org/wiki/LeNet):

In [ ]:
def conv(filters: int, padding: str) -> keras.layers.Conv2D:
  return keras.layers.Conv2D(filters=filters,
                             kernel_size=5,
                             padding=padding,
                             activation="sigmoid")


def pooling() -> keras.layers.MaxPooling2D:
  return keras.layers.MaxPooling2D()


def dense(units: int, activation: str = "sigmoid") -> keras.layers.Dense:
  return keras.layers.Dense(units, activation=activation)


le_net = keras.Sequential([
    keras.layers.Input(shape=(150, 150, 3)),
    conv(6, "same"),
    pooling(),
    conv(16, "valid"),
    pooling(),
    keras.layers.Flatten(),
    dense(120),
    dense(84),
    dense(6, activation="softmax")
], name="le_net")
le_net.summary()

le_net.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
               loss="sparse_categorical_crossentropy",
               metrics=["accuracy"])

training = le_net.fit(images,
                      labels,
                      epochs=15,
                      validation_split=0.30,
                      batch_size=128)
plot_metrics(training.history)

An implementation of [AlexNet](https://en.wikipedia.org/wiki/AlexNet):

In [ ]:
def conv(filters: int,
         kernel_size: int,
         padding: str = "same",
         strides: int = 1
         ) -> keras.layers.Conv2D:
  return keras.layers.Conv2D(filters=filters,
                             kernel_size=kernel_size,
                             strides=strides,
                             padding=padding,
                             activation="relu")


def pooling() -> keras.layers.MaxPooling2D:
  return keras.layers.MaxPooling2D(pool_size=3, strides=2)


def dropout(rate: float = 0.5) -> keras.layers.Dropout:
  return keras.layers.Dropout(rate)


def dense(units: int, activation: str = "relu") -> keras.layers.Dense:
  return keras.layers.Dense(units, activation=activation)


alex_net = keras.Sequential([
    keras.layers.Input(shape=(150, 150, 3)),
    conv(96, 11, "valid", 4),
    pooling(),
    conv(256, 5),
    pooling(),
    conv(384, 3),
    conv(384, 3),
    conv(384, 3),
    pooling(),
    keras.layers.Flatten(),
    dense(4096),
    dropout(),
    dense(4096),
    dropout(),
    dense(6, activation="softmax")
], name="alex_net")
alex_net.summary()

alex_net.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
                 loss="sparse_categorical_crossentropy",
                 metrics=["accuracy"])

training = alex_net.fit(images,
                        labels,
                        epochs=15,
                        validation_split=0.30,
                        batch_size=128)
plot_metrics(training.history)

An implementation of an [Inception](https://towardsdatascience.com/a-simple-guide-to-the-versions-of-the-inception-network-7fc52b863202) block (a complete network would need several of them). This implementation uses the [Keras functional API](https://keras.io/guides/functional_api/):

In [ ]:
def block(inputs: keras.layers.Layer) -> keras.layers.Layer:
  # conv 1x1
  conv11 = keras.layers.Conv2D(64, 1, activation="relu")(inputs)

  # conv 1x1 then conv 3x3
  conv11_3 = keras.layers.Conv2D(64, 1, activation="relu")(inputs)
  conv33_3 = keras.layers.Conv2D(64, 3, activation="relu", padding="same")(conv11_3)

  # conv 1x1 then conv 5x5
  conv11_5 = keras.layers.Conv2D(64, 1, activation="relu")(inputs)
  conv55_5 = keras.layers.Conv2D(64, 5, activation="relu", padding="same")(conv11_5)

  # max pool 3x3 strides 1x1 then conv 1x1
  max_pool = keras.layers.MaxPooling2D(3, 1, padding="same")(inputs)
  conv11_mp = keras.layers.Conv2D(64, 1, activation="relu")(max_pool)

  # concatenation
  concat = keras.layers.Concatenate()([conv11, conv33_3, conv55_5, conv11_mp])
  return concat

def inception() -> keras.models.Model:
  inputs = keras.layers.Input(shape=(150, 150, 3))
  x = block(inputs)
  x = keras.layers.Flatten()(x)
  outputs = keras.layers.Dense(6, activation="softmax")(x)


  model = keras.models.Model(inputs=[inputs], outputs=[outputs])
  model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
                loss="sparse_categorical_crossentropy",
                metrics=["accuracy"])
  model.summary()
  return model


inception_model = inception()
training = inception_model.fit(images, labels, epochs=10, validation_split=0.30)

## Performance evaluation on the test set

In the `seg_test` folder is a dataset which has never been seen during training.

We will use the `evaluate(X, y)` method of the model to evaluate the quality of our predictions on this dataset.

In [ ]:
test_images,test_labels = get_images(
    pathlib.Path("landscape") / "seg_test")
model.evaluate(test_images, test_labels, verbose=1)

## Error analysis

The confusion matrix is displayed and then misclassified images are viewed.

In [ ]:
def analyze_preds(preds, labels):
  confusion_matrix = sklearn.metrics.confusion_matrix(labels, preds)
  seaborn.heatmap(confusion_matrix,
                  annot=True,
                  fmt="d",
                  cmap="rocket_r",
                  xticklabels=label_names,
                  yticklabels=label_names)
  plt.title("Confusion matrix")
  plt.show()

  seaborn.countplot(x=list(map(lambda x: label_names[x], preds)))
  plt.title("Predicted labels distribution")
  plt.ylabel("#")
  plt.xlabel("Label")
  plt.show()


test_pred = numpy.argmax(model.predict(test_images), axis=-1)
analyze_preds(test_pred, test_labels)

In [ ]:
def plot_mistakes(predicted_class: str, true_class: str) -> None:
  print(f"Prediction : {predicted_class}, true class : {true_class}")
  mistakes = test_images[(test_pred == label_names.index(predicted_class))
                         & (test_labels == label_names.index(true_class))]
  random_indexes = numpy.random.choice(mistakes.shape[0],
                                       size=min(mistakes.shape[0], 25),
                                       replace=False)
  grid_indexes = itertools.product(range(5), repeat=2)

  _, ax = plt.subplots(5, 5, figsize=(15, 15))
  for img_index, (i, j) in zip(random_indexes, grid_indexes):
    ax[i, j].imshow(mistakes[img_index])
    ax[i, j].axis("off")
  plt.show()

In [ ]:
# Plot the images predicted as glacier when they have a mountain label
plot_mistakes("glacier", "mountain")

In [ ]:
# Plot the images predicted as glacier when they have a sea label
plot_mistakes("glacier", "sea")

In [ ]:
# Plot the predicted images building while they have a sea label
plot_mistakes("buildings", "sea")

## Transfer Learning

In [ ]:
base_model = keras.applications.EfficientNetV2B0(include_top=False,
                                                 weights="imagenet",
                                                 input_shape=(150, 150, 3))

base_model.trainable = False

model = keras.Sequential(
    [base_model,
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(1024, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(256, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(64, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(16, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Flatten(),
     keras.layers.Dense(6, activation="softmax", kernel_regularizer="l2")])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.summary()

In [ ]:
training = model.fit(images,
                     labels,
                     epochs=15,
                     validation_split=0.30,
                     batch_size=512)

plot_metrics(training.history)

In [ ]:
model.evaluate(test_images, test_labels, verbose=1)
test_preds = numpy.argmax(model.predict(test_images), axis=-1)
analyze_preds(test_preds, test_labels)
plot_mistakes("glacier", "mountain")
plot_mistakes("glacier", "sea")
plot_mistakes("buildings", "sea")

## Predicting in 'real' conditions

In the `seg_pred` folder are unannotated images. Therefore, we cannot properly evaluate the performance on this set. 

However, we can display pictures and the probabilities that our model assigns to each class.

In [ ]:
pred_images = get_images(
    pathlib.Path("landscape") / "seg_pred",
    create_labels=False)
pred_images.shape

In [ ]:
# Create the grid of subplots. We pass the figsize argument to enlarge
# the figure, which is small by default
_, ax = plt.subplots(10, 5, figsize=(30, 45))

# Pick 25 random indices, without replacement (we don't want to display the
# same image twice)
random_indexes = numpy.random.choice(pred_images.shape[0],
                                     size=(5, 5),
                                     replace=False)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    # Get the image and predict its class
    image = pred_images[img_index]
    probabilities = model.predict(image[None, ...])[0]
    predicted_class = label_names[numpy.argmax(probabilities)]

    # Display with matplotlib and its imshow function, very handy in computer
    # vision
    ax[i * 2, j].imshow(image)
    ax[i * 2, j].set_title(f"Example {img_index}")
    ax[i * 2, j].axis('off')

    # Display the prediction distribution on the row below
    ax[i * 2 + 1, j].bar(label_names, probabilities)

## Accuracy as a function of maximum likelihood

Let us now look at whether the accuracy varies significantly as a function of the maximum likelihood returned by the model.

In [ ]:
test_probabilities = model.predict(test_images)

In [ ]:
def threshold_accuracy(probabilities: numpy.ndarray,
                       labels: tf.Tensor,
                       threshold: float
                       ) -> typing.Tuple[float, float, float, float]:
  labels = labels.numpy()
  predictions = probabilities.argmax(axis=-1)
  mask_above = probabilities.max(axis=-1) > threshold
  mask_below = ~mask_above

  n_above = mask_above.sum()
  n_below = mask_below.sum()

  if n_above:
    above = (predictions[mask_above] == labels[mask_above]).sum() / n_above
  else:
    above = 1.
  if n_below:
    below = (predictions[mask_below] == labels[mask_below]).sum() / n_below
  else:
    below = 1.
  return above, below, n_above / labels.shape[0], n_below / labels.shape[0]


accuracy_above, accuracy_below, ratio_above, ratio_below = threshold_accuracy(
    test_probabilities, test_labels, 0.99)
print("Accuracy of the model's predictions with a probability above "
      f"the threshold ({ratio_above * 100:.2f}% of the data): "
      f"{accuracy_above:.2f}")
print("Accuracy of the model's predictions with a probability below "
      f"the threshold ({ratio_below * 100:.2f}% of the data): "
      f"{accuracy_below:.2f}")

xs = numpy.arange(101)
accuracies = []
ratios_above = []
for x in xs:
  accuracy, _, ratio_above, _ = threshold_accuracy(test_probabilities,
                                                   test_labels,
                                                   x / 100)
  accuracies.append(accuracy)
  ratios_above.append(ratio_above)
plt.plot(xs, accuracies, label="Accuracy")
plt.plot(xs, ratios_above, label="Recall")
plt.xlabel("Prediction threshold")
plt.title("Accuracy and recall as a function of the prediction threshold")
plt.legend()
plt.show()

## Transfer Learning with Transformers

In [ ]:
images, labels = get_images(pathlib.Path("landscape") / "seg_train",
                            size=224,
                            channels_first=True)

In [ ]:
import transformers


vit = transformers.TFAutoModel.from_pretrained(
    "google/vit-base-patch16-224-in21k", use_safetensors=False)

vit.trainable = False
inputs = keras.layers.Input((3, 224, 224))
x = keras.layers.Lambda(lambda t: t / 255)(inputs)
x = keras.layers.Normalization(mean=0.5, variance=0.5 ** 2)(x)

# Wrap the vit model call in a Lambda layer to handle the tensor type issue
x = keras.layers.Lambda(lambda x: vit(x).last_hidden_state)(x)

x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dense(64, activation="relu")(x)
x = keras.layers.Dense(32, activation="relu")(x)
x = keras.layers.Flatten()(x)
outputs = keras.layers.Dense(6, activation="softmax")(x)
transferred_transformer = keras.Model(inputs=inputs,
                                      outputs=outputs,
                                      name="transferred_transformer")
transferred_transformer.compile(optimizer="adam",
                                metrics=["accuracy"],
                                loss="sparse_categorical_crossentropy")
transferred_transformer.summary()

In [ ]:
transferred_transformer.fit(images, labels, epochs=15, validation_split=0.3, batch_size=512)

In [ ]:
test_images, test_labels = get_images(
    pathlib.Path("landscape") / "seg_test",
    size=224,
    channels_first=True)